# Credit Card Fraud Detection with Amazon SageMaker

## Overview

This notebook demonstrates a complete machine learning pipeline for credit card fraud detection using **Amazon SageMaker**. We'll build, train, and deploy a production-ready fraud detection model using XGBoost.

### What You'll Learn

1. Download and prepare a real-world fraud detection dataset
2. Handle severe class imbalance using SMOTE
3. Upload training data to Amazon S3
4. Train an XGBoost model on SageMaker infrastructure
5. Deploy the model to a real-time endpoint
6. Make predictions on live transactions
7. Clean up resources to avoid unnecessary costs

### Dataset

We'll use the **Credit Card Fraud Detection Dataset** which contains:
- 284,807 transactions over 2 days
- 492 fraudulent transactions (0.172% - highly imbalanced!)
- 30 features (Time, Amount, and 28 PCA-transformed features V1-V28)

### Estimated Costs

- **Training**: ~$0.10-0.25 (ml.m5.xlarge for 5-10 minutes)
- **Hosting**: ~$0.10/hour (ml.t2.medium)
- **S3 Storage**: < $0.01 (dataset is ~150MB)
- **Total for this workshop**: < $1.00 if cleaned up promptly

### Prerequisites

- AWS Account with SageMaker permissions
- SageMaker Studio or Notebook instance (Python 3.10)
- IAM role with SageMaker, S3, and CloudWatch permissions

---

## Step 1: Import Required Libraries and Setup

First, let's import all necessary libraries and configure our AWS environment.

In [ ]:
# Standard libraries
import pandas as pd
import numpy as np
import json
import os
import sys
from datetime import datetime
import time

# AWS SDK
import boto3
import sagemaker
from sagemaker import get_execution_role
from sagemaker.estimator import Estimator
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

# Machine Learning libraries
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Add project source to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(project_root, 'src'))

# Import our custom data processor
from data_processor import FraudDataProcessor

print("All libraries imported successfully!")
print(f"SageMaker SDK version: {sagemaker.__version__}")

## Step 2: Configure AWS and SageMaker

Set up AWS credentials, regions, and SageMaker session. We'll create a unique S3 bucket for this project.

In [ ]:
# Initialize SageMaker session
sagemaker_session = sagemaker.Session()
region = sagemaker_session.boto_region_name
role = get_execution_role()

# Get account ID for bucket naming
sts_client = boto3.client('sts')
account_id = sts_client.get_caller_identity()['Account']

# Create unique bucket name with timestamp
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
bucket_name = f"fraud-detection-{account_id}-{timestamp}"

# Create S3 bucket if it doesn't exist
s3_client = boto3.client('s3', region_name=region)
try:
    if region == 'us-east-1':
        s3_client.create_bucket(Bucket=bucket_name)
    else:
        s3_client.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={'LocationConstraint': region}
        )
    print(f"Created S3 bucket: {bucket_name}")
except s3_client.exceptions.BucketAlreadyOwnedByYou:
    print(f"Bucket {bucket_name} already exists and is owned by you")
except Exception as e:
    print(f"Error creating bucket: {e}")
    raise

# Define S3 paths
prefix = 'fraud-detection'
train_prefix = f'{prefix}/train'
validation_prefix = f'{prefix}/validation'
output_prefix = f'{prefix}/output'

train_s3_uri = f's3://{bucket_name}/{train_prefix}'
validation_s3_uri = f's3://{bucket_name}/{validation_prefix}'
output_s3_uri = f's3://{bucket_name}/{output_prefix}'

print("\n" + "="*60)
print("AWS Configuration")
print("="*60)
print(f"Region: {region}")
print(f"Account ID: {account_id}")
print(f"S3 Bucket: {bucket_name}")
print(f"Training data: {train_s3_uri}")
print(f"Validation data: {validation_s3_uri}")
print(f"Model output: {output_s3_uri}")
print(f"IAM Role: {role.split('/')[-1]}")
print("="*60)

## Step 3: Download and Explore the Dataset

We'll download the Credit Card Fraud Detection dataset and examine its characteristics.

In [ ]:
# Initialize data processor
processor = FraudDataProcessor(random_state=42)

# Define local data directory
data_dir = os.path.join(project_root, 'data')
os.makedirs(data_dir, exist_ok=True)

# Download dataset
dataset_path = os.path.join(data_dir, 'creditcard.csv')

if not os.path.exists(dataset_path):
    print("Downloading Credit Card Fraud Dataset...")
    print("This may take 1-2 minutes...\n")
    success = processor.download_dataset(
        url="https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv",
        local_path=dataset_path
    )
    if not success:
        raise Exception("Failed to download dataset")
else:
    print(f"Dataset already exists at: {dataset_path}")

# Load dataset
print("\nLoading dataset...")
df = processor.load_dataset(dataset_path)

# Display first few rows
print("\nDataset Preview:")
display(df.head())

# Display dataset info
print("\nDataset Information:")
print(df.info())

# Display statistics
print("\nDataset Statistics:")
display(df.describe())

## Step 4: Analyze Class Imbalance

Fraud detection datasets are typically highly imbalanced. Let's examine the class distribution.

In [ ]:
# Analyze class imbalance
imbalance_stats = processor.analyze_imbalance(df)

# Visualize class distribution
print("Class Distribution:")
print(f"  Legitimate: {imbalance_stats['legitimate']:,} transactions")
print(f"  Fraudulent: {imbalance_stats['fraud']:,} transactions")
print(f"  Fraud percentage: {imbalance_stats['fraud_percentage']:.3f}%")
print(f"  Imbalance ratio: {imbalance_stats['imbalance_ratio']}:1")

# Display warning if severely imbalanced
if imbalance_stats['fraud_percentage'] < 1.0:
    print("\n⚠️  SEVERE CLASS IMBALANCE DETECTED!")
    print("   This requires special handling (SMOTE, class weights, etc.)")

## Step 5: Apply SMOTE for Class Balancing

**SMOTE (Synthetic Minority Over-sampling Technique)** creates synthetic samples of the minority class to balance the dataset.

### Why SMOTE?

- Prevents model from being biased toward the majority class
- Creates synthetic examples rather than duplicating existing ones
- Improves model's ability to detect fraud

We'll target a 30% fraud ratio (down from 99.8% legitimate).

In [ ]:
# Apply SMOTE to balance the dataset
print("Applying SMOTE to balance dataset...\n")
df_balanced = processor.balance_dataset(df, sampling_strategy=0.3)

# Analyze balanced dataset
print("\nAnalyzing balanced dataset...")
balanced_stats = processor.analyze_imbalance(df_balanced)

print("\nBalancing Results:")
print(f"  Original size: {len(df):,} transactions")
print(f"  Balanced size: {len(df_balanced):,} transactions")
print(f"  Size increase: {(len(df_balanced)/len(df) - 1)*100:.1f}%")
print(f"  New fraud ratio: {balanced_stats['fraud_percentage']:.1f}%")

## Step 6: Split Data and Upload to S3

We'll split the balanced dataset into training and validation sets, then upload them to S3 for SageMaker training.

In [ ]:
# Split the data
print("Splitting data into train/validation sets...\n")

# Separate features and target
X = df_balanced.drop('Class', axis=1)
y = df_balanced['Class']

# Split into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

# Combine features and target
train_df = pd.concat([X_train, y_train], axis=1)
val_df = pd.concat([X_val, y_val], axis=1)

print(f"Training set: {len(train_df):,} transactions")
print(f"  - Fraud: {sum(y_train == 1):,} ({sum(y_train == 1)/len(y_train)*100:.1f}%)")
print(f"  - Legitimate: {sum(y_train == 0):,} ({sum(y_train == 0)/len(y_train)*100:.1f}%)")

print(f"\nValidation set: {len(val_df):,} transactions")
print(f"  - Fraud: {sum(y_val == 1):,} ({sum(y_val == 1)/len(y_val)*100:.1f}%)")
print(f"  - Legitimate: {sum(y_val == 0):,} ({sum(y_val == 0)/len(y_val)*100:.1f}%)")

# Save locally
train_path = os.path.join(data_dir, 'train.csv')
val_path = os.path.join(data_dir, 'validation.csv')

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)

print(f"\nSaved training data to: {train_path}")
print(f"Saved validation data to: {val_path}")

In [ ]:
# Upload data to S3
print("Uploading data to S3...\n")

try:
    # Upload training data
    train_s3_path = sagemaker_session.upload_data(
        path=train_path,
        bucket=bucket_name,
        key_prefix=train_prefix
    )
    print(f"Training data uploaded to: {train_s3_path}")
    
    # Upload validation data
    val_s3_path = sagemaker_session.upload_data(
        path=val_path,
        bucket=bucket_name,
        key_prefix=validation_prefix
    )
    print(f"Validation data uploaded to: {val_s3_path}")
    
    print("\n✓ Data upload successful!")
    
except Exception as e:
    print(f"Error uploading data: {e}")
    raise

## Step 7: Prepare Training Script for SageMaker

We'll package our training script and dependencies for SageMaker. The training script is located at:
- `scripts/train.py` - Main training logic
- `scripts/inference.py` - Inference handler for endpoints
- `src/data_processor.py` - Data processing utilities

In [ ]:
# Define paths to scripts
scripts_dir = os.path.join(project_root, 'scripts')
src_dir = os.path.join(project_root, 'src')

train_script = os.path.join(scripts_dir, 'train.py')
inference_script = os.path.join(scripts_dir, 'inference.py')

# Verify scripts exist
if not os.path.exists(train_script):
    raise FileNotFoundError(f"Training script not found: {train_script}")
if not os.path.exists(inference_script):
    raise FileNotFoundError(f"Inference script not found: {inference_script}")

print("Training script location:")
print(f"  {train_script}")
print("\nInference script location:")
print(f"  {inference_script}")
print("\n✓ Scripts verified!")

## Step 8: Create SageMaker Estimator

We'll create a SageMaker Estimator to handle training job orchestration.

### Training Configuration:

- **Instance Type**: ml.m5.xlarge (4 vCPUs, 16 GB RAM)
- **Framework**: Python 3.10
- **Hyperparameters**: XGBoost parameters optimized for fraud detection
- **Estimated Training Time**: 5-10 minutes
- **Estimated Cost**: $0.10 - $0.25

In [ ]:
# Define hyperparameters
hyperparameters = {
    'n-estimators': 100,        # Number of boosting rounds
    'max-depth': 6,             # Maximum tree depth
    'learning-rate': 0.1,       # Step size shrinkage
    'random-state': 42          # For reproducibility
}

# Create unique job name
job_name = f"fraud-detection-{timestamp}"

print("Creating SageMaker Estimator...\n")
print("Training Configuration:")
print("="*60)
print(f"Job name: {job_name}")
print(f"Instance type: ml.m5.xlarge")
print(f"Instance count: 1")
print(f"Max runtime: 3600 seconds (1 hour)")
print(f"Python version: 3.10")
print(f"\nHyperparameters:")
for key, value in hyperparameters.items():
    print(f"  {key}: {value}")
print("="*60)

In [ ]:
# Create the estimator
estimator = Estimator(
    entry_point='train.py',
    source_dir=scripts_dir,
    dependencies=[src_dir],
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    framework_version='3.10',
    py_version='py310',
    base_job_name='fraud-detection',
    output_path=output_s3_uri,
    hyperparameters=hyperparameters,
    max_run=3600,  # Maximum runtime in seconds (1 hour)
    use_spot_instances=False,  # Set to True to save costs with spot instances
    sagemaker_session=sagemaker_session,
    enable_network_isolation=False,
    code_location=f's3://{bucket_name}/{prefix}/code'
)

print("\n✓ SageMaker Estimator created successfully!")

## Step 9: Train the Model on SageMaker

Now we'll launch the training job on SageMaker infrastructure. This will:

1. Provision a training instance (ml.m5.xlarge)
2. Download training data from S3
3. Execute the training script
4. Save the trained model to S3
5. Terminate the training instance

**Note**: This cell will take 5-10 minutes to complete. You can monitor progress in the SageMaker console.

In [ ]:
# Define training inputs
train_input = sagemaker.inputs.TrainingInput(
    s3_data=train_s3_path,
    content_type='text/csv'
)

validation_input = sagemaker.inputs.TrainingInput(
    s3_data=val_s3_path,
    content_type='text/csv'
)

# Start training
print("Starting SageMaker training job...")
print("\nThis will take approximately 5-10 minutes.")
print("You can monitor progress in the SageMaker console.\n")
print("="*60)

start_time = time.time()

try:
    estimator.fit(
        inputs={
            'train': train_input,
            'validation': validation_input
        },
        wait=True,
        logs='All'
    )
    
    elapsed_time = time.time() - start_time
    
    print("\n" + "="*60)
    print("✓ Training completed successfully!")
    print(f"Training time: {elapsed_time/60:.1f} minutes")
    print(f"Model artifacts: {estimator.model_data}")
    print("="*60)
    
except Exception as e:
    print(f"\nError during training: {e}")
    raise

## Step 10: Deploy Model to Real-Time Endpoint

Now we'll deploy the trained model to a SageMaker endpoint for real-time predictions.

### Deployment Configuration:

- **Instance Type**: ml.t2.medium (2 vCPUs, 4 GB RAM)
- **Instance Count**: 1
- **Deployment Time**: 5-8 minutes
- **Cost**: ~$0.10/hour while running

**Important**: Remember to delete the endpoint when finished to avoid ongoing charges!

In [ ]:
# Create unique endpoint name
endpoint_name = f"fraud-detection-endpoint-{timestamp}"

print("Deploying model to SageMaker endpoint...")
print("\nDeployment Configuration:")
print("="*60)
print(f"Endpoint name: {endpoint_name}")
print(f"Instance type: ml.t2.medium")
print(f"Instance count: 1")
print(f"Estimated deployment time: 5-8 minutes")
print(f"Estimated cost: ~$0.10/hour while running")
print("="*60)
print("\nStarting deployment...\n")

start_time = time.time()

try:
    predictor = estimator.deploy(
        initial_instance_count=1,
        instance_type='ml.t2.medium',
        endpoint_name=endpoint_name,
        serializer=JSONSerializer(),
        deserializer=JSONDeserializer(),
        wait=True
    )
    
    elapsed_time = time.time() - start_time
    
    print("\n" + "="*60)
    print("✓ Deployment completed successfully!")
    print(f"Deployment time: {elapsed_time/60:.1f} minutes")
    print(f"Endpoint name: {endpoint_name}")
    print(f"Endpoint status: InService")
    print("="*60)
    
except Exception as e:
    print(f"\nError during deployment: {e}")
    raise

## Step 11: Test the Endpoint with Sample Transactions

Let's test our deployed endpoint with both legitimate and fraudulent transactions.

In [ ]:
# Helper function to format prediction results
def display_prediction(transaction, result):
    """
    Display prediction results in a human-readable format.
    """
    print("\n" + "="*60)
    print("Transaction Analysis")
    print("="*60)
    
    print(f"\nAmount: ${transaction.get('Amount', 0):.2f}")
    
    # Display prediction
    prediction = result['prediction']
    fraud_prob = result['fraud_probability']
    risk_level = result['risk_level']
    
    status = "FRAUDULENT" if prediction == 1 else "LEGITIMATE"
    status_emoji = "🚨" if prediction == 1 else "✓"
    
    print(f"\n{status_emoji} Prediction: {status}")
    print(f"Fraud Probability: {fraud_prob*100:.2f}%")
    print(f"Risk Level: {risk_level}")
    
    # Display signals
    if 'signals' in result:
        print("\nSignals Detected:")
        for signal in result['signals']:
            print(f"  • {signal}")
    
    print("="*60)

In [ ]:
# Test with a legitimate transaction from validation set
print("Testing with a LEGITIMATE transaction...")

# Get a legitimate transaction
legit_transaction = val_df[val_df['Class'] == 0].iloc[0].drop('Class').to_dict()

# Make prediction
result = predictor.predict(legit_transaction)

# Display result
display_prediction(legit_transaction, result[0])

In [ ]:
# Test with a fraudulent transaction from validation set
print("Testing with a FRAUDULENT transaction...")

# Get a fraudulent transaction
fraud_transaction = val_df[val_df['Class'] == 1].iloc[0].drop('Class').to_dict()

# Make prediction
result = predictor.predict(fraud_transaction)

# Display result
display_prediction(fraud_transaction, result[0])

In [ ]:
# Test batch predictions
print("Testing BATCH predictions with 5 transactions...\n")

# Get 5 random transactions
batch_transactions = val_df.sample(n=5, random_state=42).drop('Class', axis=1).to_dict('records')

# Make batch prediction
batch_results = predictor.predict(batch_transactions)

# Display results
print("Batch Prediction Results:")
print("="*60)
for i, (transaction, result) in enumerate(zip(batch_transactions, batch_results), 1):
    status = "FRAUD" if result['prediction'] == 1 else "LEGIT"
    prob = result['fraud_probability']
    risk = result['risk_level']
    amount = transaction.get('Amount', 0)
    
    print(f"{i}. ${amount:>8.2f} | {status:>5} | {prob*100:>6.2f}% | {risk:>8}")

print("="*60)
print("\n✓ Batch predictions successful!")

## Step 12: Invoke Endpoint Programmatically

Here's how to invoke the endpoint from your application code using boto3 directly.

In [ ]:
# Example: Invoke endpoint using boto3 runtime client
runtime_client = boto3.client('sagemaker-runtime', region_name=region)

# Create a sample transaction
sample_transaction = {
    'Time': 12345.0,
    'V1': -1.359807,
    'V2': -0.072781,
    'V3': 2.536347,
    'V4': 1.378155,
    'V5': -0.338321,
    'V6': 0.462388,
    'V7': 0.239599,
    'V8': 0.098698,
    'V9': 0.363787,
    'V10': 0.090794,
    'V11': -0.551600,
    'V12': -0.617801,
    'V13': -0.991390,
    'V14': -0.311169,
    'V15': 1.468177,
    'V16': -0.470401,
    'V17': 0.207971,
    'V18': 0.025791,
    'V19': 0.403993,
    'V20': 0.251412,
    'V21': -0.018307,
    'V22': 0.277838,
    'V23': -0.110474,
    'V24': 0.066928,
    'V25': 0.128539,
    'V26': -0.189115,
    'V27': 0.133558,
    'V28': -0.021053,
    'Amount': 149.62
}

# Invoke endpoint
response = runtime_client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType='application/json',
    Accept='application/json',
    Body=json.dumps(sample_transaction)
)

# Parse response
result = json.loads(response['Body'].read().decode())

print("Endpoint Invocation Response:")
print(json.dumps(result, indent=2))

print("\n✓ Endpoint invocation successful!")

## Step 13: Integration Code Example

Here's production-ready code you can use in your application to invoke the fraud detection endpoint.

In [ ]:
# Production-ready fraud detection function
def check_fraud(transaction_data, endpoint_name, region='us-east-1'):
    """
    Check if a transaction is fraudulent using the SageMaker endpoint.
    
    Args:
        transaction_data (dict): Transaction features
        endpoint_name (str): SageMaker endpoint name
        region (str): AWS region
        
    Returns:
        dict: Prediction result with fraud probability and risk level
    """
    client = boto3.client('sagemaker-runtime', region_name=region)
    
    try:
        response = client.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType='application/json',
            Accept='application/json',
            Body=json.dumps(transaction_data)
        )
        
        result = json.loads(response['Body'].read().decode())
        return result[0]  # Return first result (single transaction)
        
    except Exception as e:
        print(f"Error invoking endpoint: {e}")
        raise


# Example usage
print("Example: Production Integration\n")
print("Python code to integrate fraud detection:")
print("""
# In your application:
from fraud_detector import check_fraud

# When processing a transaction
transaction = {
    'Time': time_seconds,
    'V1': v1_value,
    # ... other features
    'Amount': transaction_amount
}

# Get fraud prediction
result = check_fraud(transaction, endpoint_name='YOUR_ENDPOINT_NAME')

# Act on result
if result['risk_level'] in ['HIGH', 'CRITICAL']:
    # Flag for manual review
    flag_for_review(transaction, result)
elif result['fraud_probability'] > 0.7:
    # Require additional authentication
    request_2fa(transaction)
else:
    # Process normally
    approve_transaction(transaction)
""")

print("\nEndpoint details for integration:")
print(f"  Endpoint name: {endpoint_name}")
print(f"  Region: {region}")

## Step 14: Monitor Endpoint Performance

Check endpoint metrics and status.

In [ ]:
# Get endpoint description
sm_client = boto3.client('sagemaker', region_name=region)

try:
    endpoint_info = sm_client.describe_endpoint(EndpointName=endpoint_name)
    
    print("Endpoint Status:")
    print("="*60)
    print(f"Name: {endpoint_info['EndpointName']}")
    print(f"Status: {endpoint_info['EndpointStatus']}")
    print(f"Creation time: {endpoint_info['CreationTime']}")
    print(f"Last modified: {endpoint_info['LastModifiedTime']}")
    print(f"\nEndpoint Configuration: {endpoint_info['EndpointConfigName']}")
    print("="*60)
    
    # Calculate uptime
    uptime = datetime.now(endpoint_info['CreationTime'].tzinfo) - endpoint_info['CreationTime']
    uptime_hours = uptime.total_seconds() / 3600
    estimated_cost = uptime_hours * 0.10  # Approximate cost
    
    print(f"\nUptime: {uptime_hours:.2f} hours")
    print(f"Estimated cost so far: ${estimated_cost:.4f}")
    print(f"\n⚠️  Remember to delete the endpoint to stop charges!")
    
except Exception as e:
    print(f"Error getting endpoint info: {e}")

## Step 15: Cleanup Resources

**IMPORTANT**: Delete the endpoint to stop incurring charges. The endpoint costs ~$0.10/hour while running.

This will:
1. Delete the SageMaker endpoint
2. Delete the endpoint configuration
3. Optionally clean up S3 bucket and training data

In [ ]:
# Delete the endpoint
print("Cleaning up resources...\n")

try:
    # Delete endpoint
    print(f"Deleting endpoint: {endpoint_name}")
    predictor.delete_endpoint(delete_endpoint_config=True)
    print("✓ Endpoint deleted successfully")
    
    # Delete model
    print(f"\nDeleting model: {estimator.model_name}")
    sm_client.delete_model(ModelName=estimator.model_name)
    print("✓ Model deleted successfully")
    
    print("\n" + "="*60)
    print("✓ Cleanup completed successfully!")
    print("Endpoint charges have stopped.")
    print("="*60)
    
except Exception as e:
    print(f"Error during cleanup: {e}")
    print("\nPlease manually delete resources in AWS Console:")
    print(f"  1. Endpoint: {endpoint_name}")
    print(f"  2. Model: {estimator.model_name}")

In [ ]:
# Optional: Clean up S3 bucket
# WARNING: This will delete all training data and model artifacts

CLEAN_S3 = False  # Set to True to delete S3 bucket

if CLEAN_S3:
    print("\nCleaning up S3 bucket...")
    print(f"⚠️  This will delete bucket: {bucket_name}\n")
    
    try:
        # List and delete all objects
        s3_resource = boto3.resource('s3')
        bucket = s3_resource.Bucket(bucket_name)
        bucket.objects.all().delete()
        
        # Delete bucket
        bucket.delete()
        print(f"✓ S3 bucket {bucket_name} deleted successfully")
        
    except Exception as e:
        print(f"Error deleting S3 bucket: {e}")
else:
    print("\nS3 bucket cleanup skipped.")
    print(f"To manually delete, run: aws s3 rb s3://{bucket_name} --force")
    print(f"\nS3 storage cost: ~$0.001/month for {bucket_name}")

## Summary

### What We Accomplished

1. Downloaded and analyzed the Credit Card Fraud Detection dataset
2. Applied SMOTE to handle severe class imbalance (0.17% fraud to 30% fraud)
3. Split data and uploaded to S3 for SageMaker training
4. Created a SageMaker estimator with custom training script
5. Trained an XGBoost model on SageMaker infrastructure
6. Deployed the model to a real-time endpoint
7. Tested the endpoint with single and batch predictions
8. Demonstrated production integration patterns
9. Cleaned up resources to avoid unnecessary costs

### Key Takeaways

- **Class Imbalance**: SMOTE helped balance the severely imbalanced dataset
- **SageMaker Benefits**: Managed infrastructure, automatic scaling, monitoring
- **Cost Management**: Remember to delete endpoints when not in use
- **Production Ready**: The endpoint can handle real-time predictions at scale

### Next Steps

1. **Hyperparameter Tuning**: Use SageMaker's automatic hyperparameter tuning
2. **Model Monitoring**: Set up CloudWatch alarms for endpoint metrics
3. **A/B Testing**: Deploy multiple model versions for comparison
4. **Batch Transform**: Use SageMaker Batch Transform for bulk predictions
5. **MLOps**: Integrate with CI/CD pipelines using SageMaker Pipelines

### Resources

- [SageMaker Documentation](https://docs.aws.amazon.com/sagemaker/)
- [SageMaker Python SDK](https://sagemaker.readthedocs.io/)
- [XGBoost Documentation](https://xgboost.readthedocs.io/)
- [SMOTE Paper](https://arxiv.org/abs/1106.1813)

---

**Workshop Complete!** You've successfully built and deployed a production fraud detection system on AWS SageMaker.